In [1]:
import pandas as pd
import os
import numpy as np

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Remove the flagged aphiaIDs from the parquet file

In [2]:
# Read the manual removal list
df_manual = pd.read_excel('Validate_nomenclature/validated_nomenclature_manual.xlsx')

# Get AphiaIDs marked for removal
remove_ids = df_manual[df_manual['remove'] == 'x']['AphiaID'].unique()

# Read the parquet file
df_merged = pd.read_parquet('All_data/merged_all.parquet')

# Count before
count_before = len(df_merged)

# Remove records with aphiaID in remove_ids
df_filtered = df_merged[~df_merged['aphiaID'].isin(remove_ids)]

# Count after
count_after = len(df_filtered)

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

# Save filtered parquet
df_filtered.to_parquet('Filters/merged_all_filtered.parquet')

Records before: 4079954
Records after: 4069824
Records removed: 10130


# Add acceptedAphiaID to the merged file

In [3]:
# Add acceptedAphiaID to the filtered merged parquet using AphiaID -> valid_AphiaID mapping
df_validated_final = pd.read_excel('Validate_nomenclature/validated_nomenclature_manual.xlsx')

validated_map = (
    df_validated_final[["AphiaID", "valid_AphiaID"]]
    .dropna(subset=["AphiaID"])
    .drop_duplicates(subset=["AphiaID"])
    .set_index("AphiaID")["valid_AphiaID"]
)

df_filtered["acceptedAphiaID"] = df_filtered["aphiaID"].map(validated_map)

# Move acceptedAphiaID to be immediately after aphiaID (if both exist)
cols = list(df_filtered.columns)
if "acceptedAphiaID" in cols and "aphiaID" in cols:
    cols.remove("acceptedAphiaID")
    aph_idx = cols.index("aphiaID")
    cols.insert(aph_idx + 1, "acceptedAphiaID")
    df_filtered = df_filtered[cols]

df_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print("acceptedAphiaID column added and saved to Filters/merged_all_filtered.parquet")
print("Missing acceptedAphiaID values:", df_filtered["acceptedAphiaID"].isna().sum())

acceptedAphiaID column added and saved to Filters/merged_all_filtered.parquet
Missing acceptedAphiaID values: 0


# Filter based on functional groups

In [4]:
# Read validated nomenclature file
df_validated_final = pd.read_excel("Functional_groups/functional_groups_manual.xlsx")

# Identify the AphiaID column
validated_id_col = "AphiaID"

# Get AphiaIDs marked for removal
ids_to_remove = (
    df_validated_final.loc[
        df_validated_final["remove"].astype(str).str.strip().str.lower() == "x",
        validated_id_col
    ]
    .dropna()
    .unique()
)

# Read the filtered parquet file
df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

# Remove matching records
count_before = len(df_merged_filtered)
df_merged_filtered = df_merged_filtered[~df_merged_filtered["acceptedAphiaID"].isin(ids_to_remove)].copy()
count_after = len(df_merged_filtered)

# Save back with the same name
df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4069824
Records after: 4049745
Records removed: 20079


# Repopulate taxonomy columns

In [5]:
# Load final_nomenclature.xlsx
df_nomenclature = pd.read_excel('Validate_nomenclature/validated_nomenclature_manual.xlsx')

# Create mapping from acceptedAphiaID (AphiaID in xlsx) to taxonomy columns
nomenclature_map = (
    df_nomenclature[['valid_AphiaID', 'status', 'unacceptreason', 'rank', 'phylum', 'class', 
                     'order', 'family', 'genus', 'valid_authority', 'valid_name', 
                     'isMarine', 'isBrackish']]
    .drop_duplicates(subset=['valid_AphiaID'])
    .set_index('valid_AphiaID')
)

# Clear and repopulate taxonomy columns
df_merged_filtered['taxonomicStatus'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['status']).astype('string')
df_merged_filtered['taxonomicIssue'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['unacceptreason']).astype('string')
df_merged_filtered['taxonRank'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['rank']).astype('string')
df_merged_filtered['phylum'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['phylum']).astype('string')
df_merged_filtered['class'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['class']).astype('string')
df_merged_filtered['order'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['order']).astype('string')
df_merged_filtered['family'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['family']).astype('string')
df_merged_filtered['genus'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['genus']).astype('string')
df_merged_filtered['validAuthority'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['valid_authority']).astype('string')
df_merged_filtered['acceptedScientificName'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['valid_name']).astype('string')

# Convert isMarine and isBrackish to nullable boolean
df_merged_filtered['isMarine'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['isMarine']).map({1: True, 0: False}).astype('boolean')
df_merged_filtered['isBrackish'] = df_merged_filtered['acceptedAphiaID'].map(nomenclature_map['isBrackish']).map({1: True, 0: False}).astype('boolean')

# Save the updated parquet file
df_merged_filtered.to_parquet('Filters/merged_all_filtered.parquet')

print("Taxonomy columns repopulated and saved to Filters/merged_all_filtered.parquet")
print(f"Missing taxonomicStatus: {df_merged_filtered['taxonomicStatus'].isna().sum()}")
print(f"Missing validAuthority: {df_merged_filtered['validAuthority'].isna().sum()}")


Taxonomy columns repopulated and saved to Filters/merged_all_filtered.parquet
Missing taxonomicStatus: 0
Missing validAuthority: 52130


# Taxonomic filters

In [6]:
# Remove all records where acceptedAphiaID == 2 from the filtered merged parquet

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

count_before = len(df_merged_filtered)

if "acceptedAphiaID" in df_merged_filtered.columns:
    df_merged_filtered = df_merged_filtered[df_merged_filtered["acceptedAphiaID"] != 2].copy()
else:
    raise KeyError("acceptedAphiaID column not found in df_merged_filtered")

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049745
Records after: 4049732
Records removed: 13


In [7]:
# Filter merged_all_filtered.parquet to keep only specified phyla and report counts

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "phylum" not in df_merged_filtered.columns:
    raise KeyError("phylum column not found in df_merged_filtered")

count_before = len(df_merged_filtered)

allowed_phyla = [
    "Annelida", "Mollusca", "Arthropoda", "Echinodermata", "Cnidaria", "Porifera",
    "Bryozoa", "Chordata", "Brachiopoda", "Nematoda", "Nemertea", "Tardigrada",
    "Platyhelminthes", "Hemichordata", "Kinorhyncha", "Priapulida", "Loricifera",
    "Phoronida", "Entoprocta"
]

df_merged_filtered = df_merged_filtered[df_merged_filtered["phylum"].isin(allowed_phyla)].copy()

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049732
Records after: 4049697
Records removed: 35


In [8]:
# Remove records where class equals specified classes

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "class" not in df_merged_filtered.columns:
    raise KeyError("class column not found in df_merged_filtered")

count_before = len(df_merged_filtered)

classes_to_remove = [
    "Aves", "Mammalia", "Amphibia", "Arachnida", "Chilopoda", "Diplopoda", "Hexapoda",
    "Archiacanthocephala", "Symphyla", "Cricoconarida", "Rostroconchia", "Trilobita",
    "Cestoda", "Gordioida", "Coelacanthi", "Cubozoa", "Elasmobranchii", "Eoacanthocephala",
    "Holocephali", "Monogenea", "Nuda", "Palaeacanthocephala", "Petromyzonti", "Scyphozoa",
    "Thaliacea", "Myxini", "Dipneusti", "Holostei", "Teleostei", "Chondrostei", "Sphenodontia",
    "Branchiopoda", "Leptocardii", "Phylactolaemata", "Craniata", "Myxozoa", "Tantulocarida", "Trematoda"
]

df_merged_filtered = df_merged_filtered[~df_merged_filtered["class"].isin(classes_to_remove)].copy()

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049697
Records after: 4049660
Records removed: 37


In [9]:
# Remove records where class equals specified classes

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "class" not in df_merged_filtered.columns:
    raise KeyError("class column not found in df_merged_filtered")

count_before = len(df_merged_filtered)

orders_to_remove = [
    "Squamata","Diplura","Testudines","Salmoniformes","Gadiformes","Scorpaeniformes",
    "Perciformes","Pleuronectiformes","Squaliformes","Pteropoda","Pteropoda","Copelata",
    "Stylommatophora","Osmeriformes","Flosculariaceae","Myctophiformes","Gasterosteiformes",
    "Narcomedusae","Anguilliformes","Lumbriculida","Stomiiformes","Clupeiformes","Esociformes",
    "Cypriniformes","Aulopiformes","Anostraca","Lophiiformes","Ceratitida","Notacanthiformes",
    "Arhynchobdellida","Oegopsida","Beryciformes","Belemnitida","Syngnathiformes",
    "Stephanoberyciformes","Zeiformes","Acipenseriformes","Lampriformes","Beloniformes",
    "Arguloida","Saccopharyngiformes","Ophidiiformes","Tetraodontiformes","Siphonotretida",
    "Polypteriformes","Mugiliformes","Amiiformes","Bivalvulida","Percopsiformes","Characiformes",
    "Prolecanitida","Cetomimiformes","Batrachoidiformes","Modiomorphida","Siluriformes",
    "Gobiesociformes","Osteoglossiformes","Fordillida","Millericrinida","Cyrtodontida",
    "Gymnotiformes","Cyprinodontiformes","Lepisosteiformes","Agoniatitida","Atheriniformes",
    "Encrinida","Leperditicopida","Petromyzontiformes","Orthocerida","Calanoida","Cyclopoida",
    "Gelyelloida","Misophrioida","Monstrilloida","Mormonilloida","Platycopioida","Canuelloida",
    "Siphonostomatoida","Pteropoda","Pteropoda","Euphausiacea","Paludicellidae","Spongillida",
    "Unionida","Limnomedusae","Trachymedusae","Semaeostomeae","Millericrinida"
]

df_merged_filtered = df_merged_filtered[~df_merged_filtered["order"].isin(orders_to_remove)].copy()

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049660
Records after: 4049635
Records removed: 25


In [10]:
# Remove records where family is in the specified list

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "family" not in df_merged_filtered.columns:
    raise KeyError("family column not found in df_merged_filtered")

count_before = len(df_merged_filtered)

families_to_remove = [
    "Hyperiidae","Phronimidae","Lanceolidae","Platyscelidae","Paraphronimidae",
    "Cystisomatidae","Oxycephalidae","Lycaeidae","Euphausiidae","Lumbricidae",
    "Augaptilidae","Calanidae","Discoidae","Heterorhabdidae","Metridinidae",
    "Oncaeidae","Rhincalanidae","Sagittidae","Scinidae","Scolecitrichidae",
    "Spinocalanidae","Tomopteridae","Onchulidae","Acanthogammaridae","Cambaridae",
    "Crangonyctidae","Crypturopodidae","Heterakidae","Hyalellidae","Hydridae",
    "Lymnaeidae","Oxyuridae","Planorbidae","Porcellionidae","Sinolapotamon patellifer",
    "Sphaeriidae","Trachelipodidae"
]

df_merged_filtered = df_merged_filtered[~df_merged_filtered["family"].isin(families_to_remove)].copy()

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049635
Records after: 4049635
Records removed: 0


In [11]:
# Remove records where genus is in the specified list

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "genus" not in df_merged_filtered.columns:
    raise KeyError("genus column not found in df_merged_filtered")

count_before = len(df_merged_filtered)

genera_to_remove = [
    "Tomopteris", "Pelagobia", "Phalacrophorus", "Typhloscolex", "Acartia", 
    "Aetideopsis", "Aglantha", "Augaptilus", "Bathykorus", "Benthosema", 
    "Beroe", "Botrynema", "Bradyidius", "Calocalanus", "Centropages", 
    "Chiridiella", "Chiridius", "Conchoecia", "Crossota", "Cyclocaris", 
    "Daphnia", "Dimophyes", "Dinonemertes", "Euchaeta", "Eukrohnia", 
    "Euphausia", "Fritillaria", "Gaetanus", "Homoeonema", "Lucicutia", 
    "Metridia", "Microcalanus", "Microsetella", "Oikopleura", "Onchocalanus", 
    "Paraeuchaeta", "Pseudocalanus", "Pseudochirella", "Sagitta", "Scaphocalanus", 
    "Scolecithricella", "Spinocalanus", "Temorites", "Thysanoessa", "Triconia", 
    "Xanthocalanus", "Kinonchulus", "Micoletzkya", "Pisidium", "Pyrgula", 
    "Acanthechiniscus", "Bryocamptus", "Bryodelphax", "Bryodrilus", "Chaetogaster", 
    "Cornechiniscus", "Echiniscus", "Enchytronia", "Eudorylaimus", "Euritina", 
    "Glibertia", "Harpacticella", "Helobdella", "Howeina", "Hypechiniscus", 
    "Inoceramidae", "Kikuchicamptus", "Kuiperiana", "Limnodrilus", "Moraria", 
    "Orthonotacythere", "Plectus", "Pseudechiniscus", "Pteroloxa", "Reticulipora", 
    "Rhyacodrilus", "Slavina", "Spirosperma", "Stephanosella", "Stylaria", 
    "Testechiniscus", "Uncinais", "Varichaetadrilus", "Vejdovskyella", "Wallerconcha", 
    "Urnatella"
]

df_merged_filtered = df_merged_filtered[~df_merged_filtered["genus"].isin(genera_to_remove)].copy()

count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049635
Records after: 4049635
Records removed: 0


In [12]:
# Remove records with specified acceptedScientificName values

if "df_merged_filtered" not in globals():
    df_merged_filtered = pd.read_parquet("Filters/merged_all_filtered.parquet")

if "acceptedScientificName" not in df_merged_filtered.columns:
    raise KeyError("acceptedScientificName column not found in df_merged_filtered")

remove_names = [
    "Acartia clausi","Acartia longiremis","Aetideopsis armata","Aetideopsis minor",
    "Aetideopsis multiserrata","Aetideopsis rostrata","Aglantha digitale","Aglantha digitalis",
    "Augaptilus glacialis","Augaptilus spinifrons","Beroe abyssicola","Boroecia borealis",
    "Boroecia maxima","Bosmina (Bosmina) longirostris","Botrynema brucei","Botrynema ellinorae",
    "Bougainvillia superciliaris","Bradyidius armatus","Bradyidius similis","Calanus finmarchicus",
    "Calanus glacialis","Calanus hyperboreus","Centropages hamatus","Chiridiella abyssalis",
    "Chiridiella macrodactyla","Chiridiella pacifica","Chiridiella reducta","Chiridiella reductella",
    "Chiridius gracilis","Chiridius obtusifrons","Chiridius pacificus","Clione limacina",
    "Conchoecia borealis","Conchoecia elegans","Conchoecia maxima","Conchoecia obtusata",
    "Crossota millsae","Crossota norvegica","Crystallophyes amygdalina","Cyanea capillata",
    "Cyclocaris guilelmi","Dimophyes arctica","Disco triangularis","Discoconchoecia elegans",
    "Drepanopus bungei","Euaugaptilus hyperboreus","Eucalanus bungii","Euchaeta farrani",
    "Eukrohnia bathypelagica","Eukrohnia hamata","Eurytemora richingsi","Fritillaria borealis",
    "Fritillaria polaris","Gaetanus brevirostris","Gaetanus brevispinus","Gaetanus tenuispinus",
    "Gaidius brevispinus","Gaidius tenuispinus","Gilia reticulata","Haloptilus acutifrons",
    "Haloptilus pseudooxycephalus","Heterorhabdus compactus","Heterorhabdus norvegicus",
    "Homoeonema platygonon","Hyalopontius typicus","Hymenodora glacialis","Hymenodora gracilis",
    "Hyperia galba","Hyperia medusarum","Jaschnovia tolli","Lanceola clausii","Lubbockia brevis",
    "Lubbockia glacialis","Lucicutia anomala","Lucicutia pacifica","Lucicutia polaris",
    "Lucicutia pseudopolaris","Margelopsis hartlaubii","Meganyctiphanes norvegica","Mertensia ovum",
    "Metridia longa","Metridia lucens","Metridia pacifica","Microcalanus pusillus",
    "Microcalanus pygmaeus","Microsetella norvegica","Mimocalanus crassus","Mimocalanus damkaeri",
    "Mimocalanus nudus","Mimocalanus sulcifrons","Mimonectes stephenseni","Muggiaea bargmannae",
    "Neocalanus cristatus","Oikopleura dioica","Oikopleura (Vexillaria) vanhoeffeni",
    "Oikopleura (Vexillaria) labradoriensis","Oikopleura vanhoffeni","Oithona atlantica",
    "Oithona similis","Oligobrachia haakonmosbiensis","Oncaea borealis","Oncaea compacta",
    "Oncaea delicata","Oncaea englishi","Oncaea lacinia","Oncaea minuta","Oncaea ornata",
    "Oncaea parila","Oncaea prolata","Oncaea pumilis","Onchocalanus cristatus","Paraeuchaeta barbata",
    "Paraeuchaeta glacialis","Paraeuchaeta gracilis","Paraeuchaeta norvegica","Paraeuchaeta polaris",
    "Paragotoea bathybia","Paraheterorhabdus compactus","Pelagobia longicirrata","Periphylla periphylla",
    "Phaennocalanus unispinosus","Phalacrophorus borealis","Plotocnide borealis","Proceroecia vityazi",
    "Pseudocalanus acuspes","Pseudocalanus elongatus","Pseudocalanus major","Pseudocalanus minutus",
    "Pseudochirella spectabilis","Pseudosagitta maxima","Rhabdoon reesi","Rhabdoon singulare",
    "Sagitta bipunctata","Sagitta maxima","Scaphocalanus acrocephalus","Scaphocalanus brevicornis",
    "Scaphocalanus magnus","Scaphocalanus medius","Scaphocalanus polaris","Scina borealis",
    "Scolecithricella minor","Scolecitrichopsis polaris","Sminthea arctica","Solmundella bitentaculata",
    "Spinocalanus abyssalis","Spinocalanus antarcticus","Spinocalanus brevicaudatus",
    "Spinocalanus elongatus","Spinocalanus horridus","Spinocalanus longicornis","Spinocalanus magnus",
    "Spinocalanus polaris","Spinocalanus spinosus","Temora longicornis","Temorites brevis",
    "Themisto abyssorum","Themisto compressa","Themisto gaudichaudii","Thysanoessa inermis",
    "Thysanoessa longicaudata","Thysanoessa raschii","Tomopteris helgolandica","Triconia borealis",
    "Triconia conifera","Triconia minuta","Typhloscolex muelleri","Xanthocalanus borealis",
    "Xanthocalanus hirtipes","Xanthocalanus polarsternae","Xanthocalanus profundus",
    "Xanthocalanus spinodenticulatus","Apherusa glacialis","Gammarus wilkitzkii","Onisimus glacialis",
    "Onisimus nanseni","Paracratis minuta","Spirotropis modiolus","Membranipora limbata",
    "Cristatella mucedo","Embolocephalus nikolskyi","Cernosvitoviella atrata","Clavisparsa clavata",
    "Claxtonia wendti","Cluthia cluthae","Cognettia lapponica","Cognettia varisetosa",
    "Cytheridea punctillata","Cytheropteron arcuatum","Cytheropteron biconvexa",
    "Cytheropteron champlainum","Cytheropteron sulense","Cytheropteron suzdalskyi",
    "Cytheropteron tumefactum","Escharella trispinosa","Eumonhystera minuta","Gammarus pulex",
    "Glossiphonia complanata","Glossiphonia verrucata","Henlea dicksoni","Henlea heleotropha",
    "Henlea similis","Krithe glacialis","Lepetella laterocompressa","Macoma obliqua",
    "Mancikellia divae","Marionina deminuta","Mesenchytraeus beringensis","Mesenchytraeus chaunus",
    "Mesenchytraeus falciformis","Mesenchytraeus flavus","Mesenchytraeus kontrimavichusi",
    "Mesenchytraeus primaevus","Mesenchytraeus svetae","Mesenchytraeus tundrus","Mesenteripora meandrina",
    "Metopaulias depressus","Nais communis","Nucula minima","Onychocella propinqua",
    "Paracratis minuta","Paracyprideis pseudopunctillata","Phylactella peristomata","Saccella pennula",
    "Schizoporella biaperta","Semicytherura complanata","Tasserkidrilus americanus",
    "Asellus (Asellus) aquaticus"
]

count_before = len(df_merged_filtered)
df_merged_filtered = df_merged_filtered[~df_merged_filtered["acceptedScientificName"].isin(remove_names)].copy()
count_after = len(df_merged_filtered)

df_merged_filtered.to_parquet("Filters/merged_all_filtered.parquet")

print(f"Records before: {count_before}")
print(f"Records after: {count_after}")
print(f"Records removed: {count_before - count_after}")

Records before: 4049635
Records after: 4049635
Records removed: 0
